In [ ]:
import pandas as pd
# Read all sheets from the Excel file and combine them
excel_file = "C:\\Users\\TrevorWhite\\Downloads\\mlb_pitch_usage.xlsx"
excel = pd.ExcelFile(excel_file)

# Initialize an empty list to store DataFrames
dfs = []

# Read each sheet and add sheet name as a column
for sheet_name in excel.sheet_names:
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
    df['sheet_name'] = sheet_name  # Add sheet name as a column
    dfs.append(df)

# Combine all DataFrames
df_pitch_usage = pd.concat(dfs, ignore_index=True)

In [ ]:
df_pitch_usage.head()

In [ ]:
import numpy as np
# First clean the data by removing rows where 'Pitch Type' is NaN
df_pitch_usage = df_pitch_usage.dropna(subset=['Pitch Type'])

# Define the mapping for pitch types
pitch_type_mapping = {
    'STSweeper': 'SLSlider',  # Change 'STSweeper' to 'SLSlider'
    'CUCurveball': 'CUCurveball',
    'KCKnuckle Curve': 'CUCurveball',
    'CSSlow Curve': 'CUCurveball',
    'SVSlurve': 'CUCurveball',
}

# Apply the mapping to 'Pitch Type', and retain original values for unmapped ones
df_pitch_usage['Pitch Type'] = df_pitch_usage['Pitch Type'].replace(pitch_type_mapping)



# List of columns to calculate percentile ranks for
rank_columns = ['Pitch %', 'wOBA', 'xwOBA', 'Whiff%', 'Pitcher RV / 100']

# Convert numeric columns
for col in rank_columns:
    if '%' in col:
        # Remove '%' and convert to float, handling invalid strings
        df_pitch_usage[col] = (
            df_pitch_usage[col]
            .astype(str)
            .str.replace('%', '', regex=False)
            .replace('--', np.nan)  # Replace '--' with NaN
            .astype(float) / 100
        )
    else:
        # Direct conversion for non-percentage columns
        df_pitch_usage[col] = pd.to_numeric(df_pitch_usage[col], errors='coerce')

# Calculate percentile ranks within each pitch type
for col in rank_columns:
    new_col_name = f'{col}_percentile'
    df_pitch_usage[new_col_name] = df_pitch_usage.groupby('Pitch Type')[col].rank(pct=True) * 100

# Round the percentiles
percentile_columns = [col + '_percentile' for col in rank_columns]
df_pitch_usage[percentile_columns] = df_pitch_usage[percentile_columns].round(2)

# Verify results
print("\nSample of results:")
print(df_pitch_usage[['Pitch Type'] + rank_columns + percentile_columns].head())


In [ ]:
# Invert wOBA and xwOBA percentiles
df_pitch_usage['wOBA_percentile'] = 100 - df_pitch_usage['wOBA_percentile']
df_pitch_usage['xwOBA_percentile'] = 100 - df_pitch_usage['xwOBA_percentile']

# Round to 2 decimal places
df_pitch_usage['wOBA_percentile'] = df_pitch_usage['wOBA_percentile'].round(2)
df_pitch_usage['xwOBA_percentile'] = df_pitch_usage['xwOBA_percentile'].round(2)

In [ ]:
import pandas as pd
import numpy as np

# Calculate the mean and standard deviation of 'Pitcher RV / 100'
rv_mean = df_pitch_usage['Pitcher RV / 100'].mean()
rv_std = df_pitch_usage['Pitcher RV / 100'].std()

# Calculate the z-score for 'Pitcher RV / 100'
df_pitch_usage['Pitcher RV / 100_zscore'] = (df_pitch_usage['Pitcher RV / 100'] - rv_mean) / rv_std

# Convert the z-score to a scaled value (e.g., 100 is the middle point, and higher values scale up)
df_pitch_usage['Pitcher RV / 100_scaled'] = 100 + (df_pitch_usage['Pitcher RV / 100_zscore'] * 50)

# Ensure the scaled values are rounded to a reasonable number of decimal places
df_pitch_usage['Pitcher RV / 100_scaled'] = df_pitch_usage['Pitcher RV / 100_scaled'].round(2)

# Display a sample of the scaled column



In [ ]:
df_pitch_usage.head(11)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Columns we want to correlate with Pitch % percentile
corr_columns = ['wOBA_percentile', 'xwOBA_percentile', 'Whiff%_percentile', 'Whiff%','Pitcher RV / 100_percentile', 'Pitcher RV / 100_scaled']

# Overall correlation for each column
overall_corrs = {}
for col in corr_columns:
    overall_corrs[col] = df_pitch_usage['Pitch %_percentile'].corr(df_pitch_usage[col])

# Correlations by pitch type
pitch_type_corrs = []
for pitch in df_pitch_usage['Pitch Type'].unique():
    pitch_data = df_pitch_usage[df_pitch_usage['Pitch Type'] == pitch]
    corrs = {col: pitch_data['Pitch %_percentile'].corr(pitch_data[col]) for col in corr_columns}
    corrs['Pitch Type'] = pitch
    pitch_type_corrs.append(corrs)

# Convert to DataFrame
corr_df = pd.DataFrame(pitch_type_corrs)

corr_df.head(111)


In [ ]:
print(overall_corrs)

In [ ]:

# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12))

# Plot overall correlations
pd.Series(overall_corrs).plot(kind='bar', ax=ax1)
ax1.set_title('Overall Correlations with Pitch % Percentile')
ax1.set_ylabel('Correlation Coefficient')
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax1.grid(True, alpha=0.3)

# Plot correlations by pitch type
corr_df.set_index('Pitch Type').plot(kind='bar', ax=ax2)
ax2.set_title('Correlations by Pitch Type with Pitch % Percentile')
ax2.set_ylabel('Correlation Coefficient')
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax2.grid(True, alpha=0.3)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

# Print numerical correlations
print("\nOverall Correlations with Pitch % Percentile:")
print(pd.Series(overall_corrs).round(3))

print("\nCorrelations by Pitch Type:")
print(corr_df.round(3))


In [ ]:
import statsmodels.api as sm
import numpy as np
import pandas as pd

# Step 1: Filter data for specific pitch types
filtered_pitch_types = ['FF4-Seam Fastball', 'SLSlider', 'SISinker', 'CUCurveball', 'CHChangeup']
df_pitch_usage = df_pitch_usage[df_pitch_usage['Pitch Type'].isin(filtered_pitch_types)]

# Step 2: Create dummy variables for pitch types (excluding FF4-Seam Fastball)
df_pitch_usage_with_dummies = pd.get_dummies(
    df_pitch_usage,
    columns=['Pitch Type'],
    prefix='is',
    drop_first=True  # Exclude FF4-Seam Fastball as the reference
)

# Step 3: Ensure all columns are numeric
numeric_columns = ['Whiff%', 'Pitcher RV / 100_scaled']
for col in numeric_columns:
    df_pitch_usage_with_dummies[col] = pd.to_numeric(df_pitch_usage_with_dummies[col], errors='coerce')

# Convert boolean dummy variables to integers
bool_columns = [col for col in df_pitch_usage_with_dummies.columns if col.startswith('is_')]
for col in bool_columns:
    df_pitch_usage_with_dummies[col] = df_pitch_usage_with_dummies[col].astype(int)

# Step 4: Define the features (X) and target variable (y)
dummy_columns = [col for col in df_pitch_usage_with_dummies.columns if col.startswith('is_') and col != 'is_FF4-Seam Fastball']
X = df_pitch_usage_with_dummies[['Whiff%', 'Pitcher RV / 100_scaled'] + dummy_columns]
y = df_pitch_usage_with_dummies['Pitch %']

# Step 5: Drop invalid rows
X = X.replace([np.inf, -np.inf], np.nan).dropna()
y = y.replace([np.inf, -np.inf], np.nan).dropna()

# Align X and y if rows were dropped
X, y = X.align(y, join='inner', axis=0)

# Step 6: Add a constant to the model for the intercept
X = sm.add_constant(X)

# Step 7: Fit the linear regression model
model = sm.OLS(y, X).fit()

# Step 8: Output the regression summary
print(model.summary())


In [ ]:
import statsmodels.api as sm
import pandas as pd

# Step 1: List of pitch types to analyze
pitch_types = ['FF4-Seam Fastball', 'SLSlider', 'SISinker', 'CUCurveball', 'CHChangeup']

# Step 2: Loop through each pitch type and fit a regression
for pitch_type in pitch_types:
    # Filter the dataset for the current pitch type
    filtered_data = df_pitch_usage[df_pitch_usage['Pitch Type'] == pitch_type]
    
    # Define the features (X) and target variable (y) for the regression
    X = filtered_data[['Whiff%', 'Pitcher RV / 100_scaled']]
    y = filtered_data['Pitch %']
    
    # Drop any rows with missing values in X or y
    X = X.replace([np.inf, -np.inf], np.nan).dropna()
    y = y.replace([np.inf, -np.inf], np.nan).dropna()
    
    # Align X and y in case rows were dropped
    X, y = X.align(y, join='inner', axis=0)
    
    # Add a constant to the model for the intercept
    X = sm.add_constant(X)
    
    # Fit the OLS regression model
    model = sm.OLS(y, X).fit()
    
    # Print the summary of the regression
    print(f"Regression Summary for {pitch_type}:")
    print(model.summary())
    print("\n" + "-"*80 + "\n")
